# Missing Values
---
- This notebook contains three approaches to dealing with missing values.
- Most machine learning libraries give an error if you try to build a model using data with missing values.

## Three Approaches
---
1. The Simplest Option: Drop columns with missing values.\
When the most values in the dropped columns are missing, the other way the model would lose a lot of information.

2. A Better Option: Imputation.\
Fill in the missing values with some number. For instance, the mean (average) for each column.\
Imputation it's not exactly right in most cases, but it leads better acurracy than dropping columns. This is the standard option.

3. An Extension To Imputation.\
Standard imputation fills missing values with statistics like the mean, and it usually works well, but loses the signal that data was originally absent. Imputed values may be systematically above or below their real values, in response this method add a new boolean column (`True/False` or `1/0`) as a indicator which entries were originally missing (`was_missing`), for each column with missing entries in the original dataset. So, the model learns if the presence of missing data itself carries predictive value.\
\
In some cases, this will meaningfully improve results. In other cases, it doesn't help at all.

<details>
<summary>Ver <b>¿Cuando usar Extended Imputation?</b></summary>

## ¿Cuándo te conviene usar este enfoque?
---

### 1. Cuando el dato no falta al azar (Missing Not at Random):
---
Ejemplo real: Imagina una encuesta sobre casas donde no se registró el campo precio_de_renovacion. Las casas que no llenaron ese campo probablemente nunca se renovaron (valor real 0, pero si imputas con la media vas a inflar su valor). Agregar la columna precio_de_renovacion_was_missing = True le avisa al modelo: "Ojo, este grupo de casas tiene esa característica ausente por una razón en particular".

Ejemplo en finanzas: Un cliente que omite declarar su salario mensual al pedir un préstamo podría tener ingresos inestables. Que el dato sea nulo es en sí mismo una señal de riesgo.

### 2. Cuando la imputación simple "miente" o distorsiona el dato real:
---
La imputación tradicional pone la media o mediana a todos los huecos. Esto ayuda a que Scikit-Learn no arroje un error, pero "aplana" la realidad. La columna indicadora le permite al árbol de decisión (DecisionTree / RandomForest) aprender dos reglas separadas:

- Regla A: "Si la columna was_missing es False, usa el valor imputado normalmente."

- Regla B: "Si la columna was_missing es True, ajusta o corrige la predicción porque la ausencia del dato cambia el comportamiento."

## ¿Cómo saber si funcionó en la práctica?
---
Como bien dice el texto ("In some cases, this will meaningfully improve results. In other cases, it doesn't help at all"), no hay una regla mágica previa.

En MLOps / Machine Learning, la manera de saber si debiste usar este método es mediante experimentación (validación cruzada / MAE):

- Entrenas tu modelo usando Imputación Simple y mides el MAE.

- Entrenas el modelo usando Imputación con Columna Indicadora y mides el MAE.

Si el MAE baja significativamente con el segundo enfoque, significa que las filas con valores nulos contenían un patrón importante que el modelo supo aprovechar.
</details>

## Example
---

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the data
data = pd.read_csv('../input/melbourne-housing-snapshot/melb_data.csv')

# Select target
y = data.Price

# To keep things simple, we'll use only numerical predictors
melb_predictors = data.drop(['Price'], axis=1)
X = melb_predictors.select_dtypes(exclude=['object'])

# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [3]:
X_train.head()

,Rooms,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
12167,1,5.0,3182.0,1.0,1.0,1.0,0.0,NaN,1940.0,-37.85984,144.9867,13240.0
6524,2,8.0,3016.0,2.0,2.0,1.0,193.0,NaN,NaN,-37.85800,144.9005,6380.0
8413,3,12.6,3020.0,3.0,1.0,1.0,555.0,NaN,NaN,-37.79880,144.8220,3755.0
2919,3,13.0,3046.0,3.0,1.0,1.0,265.0,NaN,1995.0,-37.70830,144.9158,8870.0
6043,3,13.3,3020.0,3.0,1.0,2.0,673.0,673.0,1970.0,-37.76230,144.8272,4217.0


Define Function to Measure Quality of Each Approach

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

### Score from Approach 1 (Drop Columns with Missing Values)
---
Since we are working with both training and validation sets, we are careful to drop the same columns in both DataFrames.

In [5]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns
                     if X_train[col].isnull().any()]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop columns with missing values):
183550.22137772635


### Score from Approach 2 (Imputation)
---
Next, we use `SimpleImputer` to replace missing values with the mean value along each column.
\
Although it's simple, filling in the mean value generally performs quite well (but this varies by dataset). While statisticians have experimented with more complex ways to determine imputed values (such as regression imputation, for instance), the complex strategies typically give no additional benefit once you plug the results into sophisticated machine learning models.

In [6]:
from sklearn.impute import SimpleImputer

# Imputation
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE from Approach 2 (Imputation):
178166.46269899711


### ¿Siempre se reemplaza con la media?
---
No. La media es la opción por defecto en SimpleImputer, pero puedes cambiar la estrategia utilizando el argumento strategy: 

```python
my_imputer = SimpleImputer(strategy='median')
```


|Estrategia|¿Cómo funciona?|¿Cuándo usarla?|
| --- | --- | --- |
|'mean' (Por defecto)|Reemplaza con la media (promedio).|En datos numéricos sin valores extremos (outliers).|
|'median'|Reemplaza con la mediana (el valor central).|En datos numéricos con valores extremos (ej. precios de casas o salarios).|
|'most_frequent'|Reemplaza con la moda (el valor más común).|Ideal para columnas categóricas o enteros discretos.|
|'constant'|Reemplaza con un valor fijo definido por ti mediante fill_value.|"Cuando quieres rellenar con un valor específico (ej. fill_value=0 o ""Desconocido"")."|

### Score from Approach 3 (An Extension to Imputation)
---

In [7]:
# Make copy to avoid changing original data (when imputing)
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))

MAE from Approach 3 (An Extension to Imputation):
178927.503183954


# Conclusion
---
So, why did imputation perform better than dropping the columns?
The training data has **10864** rows, where three columns contains missing data. For each column, **less than half** of the entries are missing. Thus, dropping the columns removes a lot of useful information, and so it makes sense that imputation would perform better.

In [8]:
# Shape of training data (num_rows, num_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(10864, 12)
Car               49
BuildingArea    5156
YearBuilt       4307
dtype: int64


Therefore (and as is common) imputation yielded better results than dropping columns.